# RouteOptimizer demo

Open-source **mine fleet** stack from [Alio IT](https://www.alioit.com/route-optimizer):

- **ML dispatch policy** on [Hugging Face](https://huggingface.co/albones80/route-optimizer)
- **A\* dispatch** and **PuLP/CBC shift plan**

This notebook installs the package, runs a small sample, and loads the learned policy.

In [ ]:
%pip install -q route-optimizer

In [ ]:
import json
import tempfile
from pathlib import Path

import route_optimizer
from route_optimizer.cli import cmd_dispatch, cmd_shift_plan
from route_optimizer.policy import load_policy
from route_optimizer import policy as policy_mod

pkg = Path(route_optimizer.__file__).resolve().parent
# Samples ship in the GitHub repo; clone a thin copy for the notebook if needed.
!git clone --depth 1 https://github.com/AlfonsoALIO/RouteOptimizer.git /tmp/RouteOptimizer 2>/dev/null || true
sample = Path("/tmp/RouteOptimizer/samples/tiny_feasible.json")
print("sample exists:", sample.is_file())

In [ ]:
class Args:
    def __init__(self, **kw):
        self.__dict__.update(kw)

out = Path("/tmp/dispatch.json")
code = cmd_dispatch(Args(
    scenario=str(sample),
    output=str(out),
    segments=8,
    trucks=2,
    time_limit=30,
))
payload = json.loads(out.read_text())
print("exit", code)
print(json.dumps({k: payload.get(k) for k in ("status", "optimal", "cost_minutes", "stop_reason", "num_segments", "num_trucks")}, indent=2))

In [ ]:
out2 = Path("/tmp/shift_plan.json")
code2 = cmd_shift_plan(Args(
    scenario=str(sample),
    output=str(out2),
    segments=4,
    time_limit=30,
))
plan = json.loads(out2.read_text())
print("exit", code2)
print(json.dumps({k: plan.get(k) for k in ("status", "optimal", "proven_optimal", "time_limit_reached", "num_segments")}, indent=2))

In [ ]:
policy = load_policy()
cfg = json.loads(Path(policy_mod.DEFAULT_WEIGHTS).with_name("dispatch_policy_config.json").read_text())
print("model_type:", cfg.get("model_type"))
print("holdout_decision_accuracy:", cfg.get("holdout_decision_accuracy"))
print("holdout_greedy_baseline_accuracy:", cfg.get("holdout_greedy_baseline_accuracy"))
print("note:", cfg.get("note"))